# 1주차. 나나를 깨우다

**부제:** Nana 개인 일정 tool: 생성·조회·삭제

## 0. 목표

이번 주에는 Nana가 메모리 저장소에 내 개인 일정을 만들고 조회/삭제하는 tool 호출 흐름을 관찰한다.

성취 기준:
- `Agent`의 `tool_call` 즉 `function calling`에 대해 이해한다.
- `tool_call`과 `tool_result`를 구분해 설명할 수 있다.
- `personal_create_schedule`과 `personal_list_schedules` trace를 찾아 사용자 요청과 맞는지 확인할 수 있다.
- 답변 문구보다 personal tool arguments와 메모리 저장소 결과를 먼저 검증할 수 있다.

![graph](./imgs/graph.jpg)

![CA](imgs/CA.jpg)

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 프록시 서버를 통해 모델 API를 호출한다. 프록시 토큰, URL, 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → 노트북의 기본 개념 실습 → 카나메이트 확장 예제 순서로 진행한다.


In [8]:
# 흐름: repo 위치와 .env를 잡고, 모든 실습 셀이 함께 쓰는 모델/trace helper를 준비한다.
# 학생은 final_text보다 extract_tool_trace 결과를 먼저 확인한다.
import json
import os
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    # 현재 폴더부터 부모 폴더로 올라가며 repo 루트를 찾는다.
    # 노트북을 notebook/ 안에서 실행해도, repo 루트에서 실행해도 같은 경로를 쓰기 위한 장치다.
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


# 앞으로 나오는 상대 경로는 모두 이 repo 루트를 기준으로 맞춘다.
REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    # 노트북이 repo 루트의 설정 파일을 찾을 수 있게 경로를 추가한다.
    sys.path.insert(0, str(REPO_ROOT))
# 파일 저장/DB 생성 위치가 흔들리지 않도록 작업 폴더를 repo 루트로 고정한다.
os.chdir(REPO_ROOT)


In [9]:
from dotenv import dotenv_values, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
# 이 노트북에서 쓰는 프록시/모델 설정값은 repo 루트의 .env 파일에서만 읽는다.
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)
ENV_VALUES = dotenv_values(ENV_PATH)


def required_env(name: str) -> str:
    value = (ENV_VALUES.get(name) or "").strip()
    if not value or value == "여기에 api key 입력":
        raise RuntimeError(f"repo 루트 .env 파일에 {name}을 설정한 뒤 다시 실행하세요.")
    return value


PROXY_TOKEN = required_env("PROXY_TOKEN")
PROXY_URL = required_env("PROXY_URL")
EMBEDDING_PROXY_URL = required_env("EMBEDDING_PROXY_URL")
OPENAI_MODEL = required_env("OPENAI_MODEL")
OPENAI_EMBEDDING_MODEL = required_env("OPENAI_EMBEDDING_MODEL")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    # temperature=0은 같은 입력에서 비슷한 tool 선택/구조화 결과가 나오게 하기 위한 설정이다.
    return ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=PROXY_TOKEN,
        base_url=PROXY_URL,
        temperature=0,
        max_completion_tokens=max_tokens,
    )


def show_json(value: Any) -> None:
    # ensure_ascii=False로 한글 payload를 사람이 읽기 쉬운 그대로 출력한다.
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    # agent 실행 결과에서 마지막 message가 사용자에게 보이는 최종 답변이다.
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    # LangChain message 전체를 그대로 보여주면 복잡하므로, 수업에서 볼 핵심만 뽑는다.
    trace = []
    for message in agent_result.get("messages", []):
        # tool_calls는 모델이 "이 도구를 이 인자로 실행해줘"라고 요청한 기록이다.
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            # type == "tool"인 message는 실제 tool 실행 결과다.
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace

## 2. 개념

오늘의 큰 그림: 자연어 요청이 들어오면 모델은 Python 함수를 직접 실행하지 않고, 호출할 도구 이름과 arguments를 구조화해서 만든다. 수업 코드는 그 tool call을 실행하고 tool result를 다시 모델에게 전달한다.

반드시 이해할 것:

- `tool_call`: 모델이 어떤 tool을 어떤 arguments로 호출하겠다고 정한 기록
- `tool_result`: 실제 Python tool 실행 결과
- 일정 생성 성공 여부는 최종 답변보다 trace와 저장 목록으로 확인한다.

막혔을 때 볼 trace: `personal_create_schedule`의 arguments, `personal_list_schedules`의 result, `created_schedule`의 `id/date/start_time`.


![AA](imgs/AA.jpg)

![tool_call](imgs/tool_calls.jpg)

## 3. 기본 개념 실습

가장 작은 흐름은 "자연어 요청 → 일정 생성 tool call → 저장된 일정 확인"이다. 여기서는 생성 도구 하나만 등록해 모델이 어떤 arguments를 만드는지 본다.


In [10]:
# 흐름: 첫 번째 tool인 personal_create_schedule을 만들고, agent가 이 tool을 호출하는지 확인한다.
# schedules 리스트는 실제 DB 대신 상태 변화를 눈으로 보기 위한 작은 저장소다.
# 실제 서비스라면 DB에 저장하지만, 1주차에서는 흐름을 단순화하려고 메모리 list를 쓴다.
schedules: list[dict[str, Any]] = []
next_schedule_number = 1

@tool("personal_create_schedule", description="개인 일정을 생성한다. date는 YYYY-MM-DD, start_time은 HH:MM 형식이다.")
def personal_create_schedule(title: str, date: str, start_time: str, attendees: list[str] | None = None) -> str:
    """Create a personal schedule."""
    # 모델이 만든 tool 인자들을 하나의 schedule dict로 묶고, tool 안에서 바로 저장한다.
    global next_schedule_number
    schedule = {
        "id": f"schedule-{next_schedule_number}",
        "title": title,
        "date": date,
        "start_time": start_time,
        "attendees": attendees or [],
    }
    next_schedule_number += 1
    schedules.append(schedule)
    # tool은 실행 결과를 JSON 문자열로 돌려주고, agent는 이 값을 보고 최종 답변을 만든다.
    return json.dumps({"ok": True, "schedule": schedule}, ensure_ascii=False)

# create_agent는 모델, tool 목록, system_prompt를 묶어 실행 가능한 agent를 만든다.
nana_basic_agent = create_agent(
    model=make_model(),
    tools=[personal_create_schedule],
    system_prompt="너는 개인 일정 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 일정 생성이 필요하면 반드시 personal_create_schedule 도구를 호출한 뒤 짧게 답한다.",
)

student_request = "내일 10시에 민수와 회의 일정 잡아줘"
# invoke에 messages를 넣으면 agent가 필요할 때 personal_create_schedule tool을 호출한다.
result = nana_basic_agent.invoke({"messages": [{"role": "user", "content": student_request}]})

print(final_text(result))
# trace는 모델이 어떤 인자로 tool을 불렀는지 보여준다.
show_json(extract_tool_trace(result))
# schedules는 tool 실행 뒤 실제 메모리 상태가 바뀌었는지 보여준다.
show_json(schedules)


내일 10시에 민수와 회의 일정을 잡아두었어요.
[
  {
    "event": "tool_call",
    "tool_name": "personal_create_schedule",
    "arguments": {
      "title": "민수와 회의",
      "date": "2026-04-24",
      "start_time": "10:00",
      "attendees": [
        "민수"
      ]
    }
  },
  {
    "event": "tool_result",
    "tool_name": "personal_create_schedule",
    "content": "{\"ok\": true, \"schedule\": {\"id\": \"schedule-1\", \"title\": \"민수와 회의\", \"date\": \"2026-04-24\", \"start_time\": \"10:00\", \"attendees\": [\"민수\"]}}"
  }
]
[
  {
    "id": "schedule-1",
    "title": "민수와 회의",
    "date": "2026-04-24",
    "start_time": "10:00",
    "attendees": [
      "민수"
    ]
  }
]


## 4. 카나메이트 확장 예제

기본 일정 생성에 `personal_list_schedules` 조회 기능을 붙인다. 이제 나나는 일정을 만들고, 같은 저장소에서 현재 일정 목록도 읽을 수 있다.


In [11]:
# 흐름: 두 번째 tool인 personal_list_schedules를 추가해, 생성된 일정이 상태에 남았는지 확인한다.
# agent가 가진 tool 목록이 늘어나면 모델이 상황에 맞게 tool을 선택한다.
@tool("personal_list_schedules", description="현재 생성된 개인 일정 목록을 조회한다.")
def personal_list_schedules() -> str:
    """List personal schedules."""
    # 조회 tool은 저장된 상태를 바꾸지 않고 현재 목록만 payload로 돌려준다.
    return json.dumps({"ok": True, "schedules": schedules}, ensure_ascii=False)

# 확장 실습 5: 등록된 개인 일정을 삭제하는 LangChain tool을 만든다.
# 구현 예시 5:
@tool("personal_delete_schedule", description="schedule_id와 일치하는 개인 일정을 삭제한다. 예: schedule-1")
def personal_delete_schedule(schedule_id: str) -> str:
    """Delete a personal schedule by id."""
    # 확장 실습 6: schedule_id와 일치하는 일정을 찾아 schedules에서 바로 삭제한다.
    # 구현 예시 6:
    deleted_schedule = None
    for index, schedule in enumerate(schedules):
        if schedule["id"] == schedule_id:
            deleted_schedule = schedules.pop(index)
            break
    # 확장 실습 7: 삭제 성공 여부와 삭제된 일정을 JSON 문자열 payload로 반환한다.
    # 구현 예시 7:
    return json.dumps(
        {"ok": deleted_schedule is not None, "deleted_schedule": deleted_schedule, "schedule_id": schedule_id},
        ensure_ascii=False,
    )

nana_agent = create_agent(
    model=make_model(),
    # 확장 실습 8: agent가 사용할 수 있는 tool 목록에 생성/조회/삭제 tool을 모두 넣는다.
    # 구현 예시 8:
    tools=[personal_create_schedule, personal_list_schedules, personal_delete_schedule],
    system_prompt=(
        "너는 개인 일정 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. "
        "일정 생성, 조회, 삭제가 필요하면 반드시 알맞은 도구를 호출한 뒤 짧게 답한다. "
        "삭제 요청은 사용자가 말한 schedule_id를 personal_delete_schedule 도구에 전달한다."
    ),
)

extension_request = "지금까지 만든 일정 목록 보여줘"
extension_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_request}]})

print(final_text(extension_result))
show_json(extract_tool_trace(extension_result))


지금까지 만든 일정은 "민수와 회의"가 2026년 4월 24일 10시에 있습니다. 다른 일정도 확인하거나 추가할까요?
[
  {
    "event": "tool_call",
    "tool_name": "personal_list_schedules",
    "arguments": {}
  },
  {
    "event": "tool_result",
    "tool_name": "personal_list_schedules",
    "content": "{\"ok\": true, \"schedules\": [{\"id\": \"schedule-1\", \"title\": \"민수와 회의\", \"date\": \"2026-04-24\", \"start_time\": \"10:00\", \"attendees\": [\"민수\"]}]}"
  }
]


## 5. 확장 예제 실행

같은 agent로 새 일정을 하나 더 만들고, 곧바로 목록 조회를 실행한다. 수강생은 요청 문장을 바꿔보며 생성 trace와 조회 trace가 어떻게 달라지는지 확인한다.


In [12]:
# 흐름: 생성, 목록 조회, 삭제 요청을 연달아 실행해 tool call trace와 저장 상태를 비교한다.
# assert는 모델 문구가 아니라 tool_result payload가 맞는지 점검한다.
extension_create_request = "내일 14시에 지아와 체크인 일정 잡아줘"
extension_create_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_create_request}]})
# 생성 요청 trace에는 personal_create_schedule tool_call/tool_result가 있어야 한다.
extension_create_trace = extract_tool_trace(extension_create_result)

extension_list_request = "현재 일정 목록 보여줘"
extension_list_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_list_request}]})
# 목록 요청 trace에는 personal_list_schedules tool_call/tool_result가 있어야 한다.
extension_list_trace = extract_tool_trace(extension_list_result)

# 삭제 요청은 schedule_id를 명시해 personal_delete_schedule tool이 어떤 일정을 지울지 알 수 있게 한다.
target_schedule_id = schedules[-1]["id"]
extension_delete_request = f"{target_schedule_id} 일정 삭제해줘"
extension_delete_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_delete_request}]})
# 삭제 요청 trace에는 personal_delete_schedule tool_call/tool_result가 있어야 한다.
extension_delete_trace = extract_tool_trace(extension_delete_result)

print(final_text(extension_create_result))
show_json(extension_create_trace)
print(final_text(extension_list_result))
show_json(extension_list_trace)
print(final_text(extension_delete_result))
show_json(extension_delete_trace)
show_json(schedules)

assert any(event["event"] == "tool_call" and event["tool_name"] == "personal_create_schedule" for event in extension_create_trace)
assert any(event["event"] == "tool_call" and event["tool_name"] == "personal_list_schedules" for event in extension_list_trace)
assert any(event["event"] == "tool_call" and event["tool_name"] == "personal_delete_schedule" for event in extension_delete_trace)
assert all(schedule["id"] != target_schedule_id for schedule in schedules)
print("1주차 확장 예제 실행 통과")


내일 14시에 지아와 체크인 일정을 잡았습니다.
[
  {
    "event": "tool_call",
    "tool_name": "personal_create_schedule",
    "arguments": {
      "title": "지아와 체크인",
      "date": "2026-04-24",
      "start_time": "14:00",
      "attendees": [
        "지아"
      ]
    }
  },
  {
    "event": "tool_result",
    "tool_name": "personal_create_schedule",
    "content": "{\"ok\": true, \"schedule\": {\"id\": \"schedule-2\", \"title\": \"지아와 체크인\", \"date\": \"2026-04-24\", \"start_time\": \"14:00\", \"attendees\": [\"지아\"]}}"
  }
]
현재 일정은 다음과 같습니다.
1. 민수와 회의 - 2026-04-24 10:00 (참석자: 민수)
2. 지아와 체크인 - 2026-04-24 14:00 (참석자: 지아)

다른 일정도 확인하거나 추가, 삭제 원하시면 말씀해 주세요.
[
  {
    "event": "tool_call",
    "tool_name": "personal_list_schedules",
    "arguments": {}
  },
  {
    "event": "tool_result",
    "tool_name": "personal_list_schedules",
    "content": "{\"ok\": true, \"schedules\": [{\"id\": \"schedule-1\", \"title\": \"민수와 회의\", \"date\": \"2026-04-24\", \"start_time\": \"10:00\", \"attendees\": [\"민수\"]}, {\

## 6. 회고

개념 확인 질문:

1. `tool_call`과 `tool_result`는 각각 누가 만들고 무엇을 의미하는가?
2. 사용자가 "내일 10시"라고 말했을 때 사람이 검증해야 할 arguments는 무엇인가?
3. 최종 답변이 자연스러워도 `personal_list_schedules` trace를 확인해야 하는 이유는 무엇인가?

작은 응용 과제: 참석자가 없는 개인 일정 요청과 참석자가 있는 일정 요청을 각각 실행하고, `attendees` arguments가 어떻게 달라지는지 비교한다.
